# SPIN Healthcare — Milestone 2, Step 3: generate answers (baseline + SPIN)

Generates answers for the healthcare fairness (subset-tagged) and privacy eval questions, for the
**baseline** and **SPIN** (coupled set zeroed) models — the inputs to the Step 4 MD-Judge awareness scorer.
**Identical generation protocol to the reproduction's Step 3:** greedy decoding, `max_new_tokens=64`,
raw-question (no chat template), single baseline computed first then suppression applied, checkpointed per
batch (resumable). Only the eval data changes (healthcare).

**The fairness CSV keeps the `subset` column**, so Step 4 can report awareness per subset AND pooled.

**Inputs (add Step-2 output as a Kaggle Dataset input):** `eval_fair_health.csv`, `eval_priv_health.csv`,
and the coupled indices (`coupled_indices_health.json`, or `health_coupled_ckpt.pt` as fallback). Internet
ON (model). GPU **T4×2**. ~3–4 h on the full eval sets (1,416 fairness + 657 privacy × baseline/SPIN);
**resumable** — a timeout/crash re-runs from the checkpoint. **Outputs:** `answers_health_fairness.csv`
(q, subset, baseline, spin), `answers_health_privacy.csv` (q, baseline, spin), `results_health_step3.json`.

## Step 0 — config + coupled indices

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, json, gc, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
N_EVAL=None                # None = full eval sets. Set an int to subsample (debug).
USE_CHAT_TEMPLATE=False    # authors feed RAW questions (verified)
GEN_BS=4
MAXNEW=64                  # eval_salad SamplingParams(max_tokens=64)
DTYPE=torch.float16

def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
CP=find('coupled_indices_health.json')
if CP:
    obj=json.load(open(CP)); COUPLED=obj['coupled'] if 'coupled' in obj else obj
    print('coupled indices from', CP)
else:
    CK2=find('health_coupled_ckpt.pt'); assert CK2, 'add coupled_indices_health.json or health_coupled_ckpt.pt as input'
    COUPLED=torch.load(CK2, map_location='cpu')['coupled']; print('coupled indices from checkpoint', CK2)
COUPLED={k:[int(x) for x in v] for k,v in COUPLED.items()}
print('coupled matrices', len(COUPLED), '| coupled weights', sum(len(v) for v in COUPLED.values()))

## Step 1 — eval data + model

In [ ]:
EF=find('eval_fair_health.csv'); EP=find('eval_priv_health.csv')
assert EF and EP, 'add Step-2 output (eval_fair_health.csv, eval_priv_health.csv) as input'
fair_df=pd.read_csv(EF); priv_df=pd.read_csv(EP)
if N_EVAL: fair_df=fair_df.iloc[:N_EVAL].copy(); priv_df=priv_df.iloc[:N_EVAL].copy()
fair_q=fair_df['question'].astype(str).tolist(); fair_sub=fair_df['subset'].astype(str).tolist()
priv_q=priv_df['question'].astype(str).tolist()
print('fairness', len(fair_q), '(subsets:', dict(pd.Series(fair_sub).value_counts()), ') | privacy', len(priv_q))

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV)

def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
def suppress(coupled):
    d=dict(model.named_parameters()); nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); p.data.view(-1)[idx]=0.0; nz+=len(idxlist)
    return nz

## Step 2 — generate (checkpointed, resumable): baseline first, then SPIN

In [ ]:
CK='/kaggle/working/gen_ckpt_health.json'
state=json.load(open(CK)) if os.path.exists(CK) else {}

@torch.no_grad()
def gen_set(key, queries):
    done=state.get(key, []); t0=time.time()
    for i in range(len(done), len(queries), GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[key]=done; json.dump(state, open(CK,'w'))
        print(f'    {key} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)', end='\r')
    print(); return done

print('baseline generation...')
base_fair=gen_set('base_fair', fair_q); base_priv=gen_set('base_priv', priv_q)
nz=suppress(COUPLED); print('applied SPIN: zeroed', nz)
print('SPIN generation...')
spin_fair=gen_set('spin_fair', fair_q); spin_priv=gen_set('spin_priv', priv_q)

pd.DataFrame({'q':fair_q,'subset':fair_sub,'baseline':base_fair,'spin':spin_fair}).to_csv('/kaggle/working/answers_health_fairness.csv',index=False)
pd.DataFrame({'q':priv_q,'baseline':base_priv,'spin':spin_priv}).to_csv('/kaggle/working/answers_health_privacy.csv',index=False)

def changed(a,b): return int(sum(1 for x,y in zip(a,b) if x.strip()!=y.strip()))
def empties(a): return int(sum(1 for x in a if not x.strip()))
res={'model':MODEL_ID,'coupled_zeroed':int(nz),'n_fair':len(fair_q),'n_priv':len(priv_q),
     'answer_changed_fair':changed(base_fair,spin_fair),'answer_changed_priv':changed(base_priv,spin_priv),
     'empty_spin_fair':empties(spin_fair),'empty_spin_priv':empties(spin_priv)}
json.dump(res, open('/kaggle/working/results_health_step3.json','w'), indent=2)
print('\nSAVED answers_health_{fairness,privacy}.csv + results_health_step3.json')
print('  answers changed by SPIN: fair %d/%d, priv %d/%d | empty spin: fair %d, priv %d'%(
    res['answer_changed_fair'],len(fair_q),res['answer_changed_priv'],len(priv_q),
    res['empty_spin_fair'],res['empty_spin_priv']))
print('Next: Step 4 MD-Judge awareness (per subset + pooled), using these CSVs.')